# Time-Series Random Forest with Grid Search: Full Data Variant

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This is a variant of the notebook `06_TimeSeries_RandomForest_GridSearch`. It trains a **random forest** for `Label_03` (**1 = Production, 0 = Changeover**) on the full, unscaled, original dataset with a chronological split, and differs in these points:

- no `StandardScaler` (a random forest does not need one);
- the piece counter `Stuckzahl` is removed (as in the other notebooks);
- the grid additionally tunes `max_features` and uses larger leaf sizes;
- the cross-validation uses `TimeSeriesSplit` with 4 splits.

Steps: sort the rows chronologically, use the first 80 % for training and the last 20 % for testing, tune with `GridSearchCV` (selection by F1-score) and evaluate on the test part. The target is excluded from the features explicitly and checked by an assertion.

## Workflow

This notebook reads the original dataset file, not the output of notebook 01. The thesis describes the model in Sections 3.2 and 4.3.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads the original dataset file, saved as `data.csv` in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Label definition.** `Label_03` is derived from the machine's own process state (Production = the machine is in process), and the NC/PLC signals describe that state directly. Very high scores can therefore partly reflect how the label is defined, not only a learned pattern. The feature-importance table (Section 6) shows which signals drive the predictions; check it for signals that are almost a copy of the label.
- **One machine, one laboratory.** The data comes from a single machine and three products; the scores say little about other machines.
- **Missing values.** The random forest handles missing values natively (scikit-learn 1.4 or newer); no imputation is applied.
- **Earlier version.** An earlier version of this notebook accidentally kept `Label_03` among the features, which produced a meaningless perfect score. This is fixed. Results have to be re-generated by running the notebook.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../data.csv")     # original dataset, see the project README
OUTPUT_DIR = Path("../outputs")     # local results, not for upload

TARGET_COL = "Label_03"
TIME_COL = "time"
TRAIN_FRACTION = 0.80
SEED = 42

## 2. Load the data and sort it chronologically

The rows are sorted by `time` *before* the features are built, so the split really is chronological.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Download data_v1_0_1.csv from https://doi.org/10.5281/zenodo.15735480, "
        "save it as data.csv in the project root, or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

for col in (TIME_COL, TARGET_COL):
    if col not in df.columns:
        raise KeyError(f"Column {col!r} not found. A chronological split needs a time column and a target.")

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
print("Time range:", df[TIME_COL].min(), "->", df[TIME_COL].max())

## 3. Build the feature matrix and the chronological split

Features are all numeric columns except the target, the other label columns, `Stuckzahl` and `time`.

In [ ]:
other_labels = [c for c in df.columns if c.startswith("Label_") and c != TARGET_COL]
drop_cols = [c for c in other_labels + [TARGET_COL, "Stuckzahl", TIME_COL] if c in df.columns]
print("Excluded from the features:", drop_cols)

X_full = df.drop(columns=drop_cols).select_dtypes(include=[np.number]).copy()
y_full = df[TARGET_COL].astype(int).to_numpy()

# Safeguards against target leakage
assert TARGET_COL not in X_full.columns, "The target must not be part of the features."
assert not any(c.startswith("Label_") for c in X_full.columns), "No label column may be a feature."

n_train = int(TRAIN_FRACTION * len(X_full))
X_train, X_test = X_full.iloc[:n_train], X_full.iloc[n_train:]
y_train, y_test = y_full[:n_train], y_full[n_train:]
feature_names = X_full.columns.tolist()

print(f"Feature matrix: {X_full.shape[0]:,} rows × {len(feature_names)} features")
print(f"Train: {len(X_train):,}  class counts {np.bincount(y_train).tolist()}")
print(f"Test:  {len(X_test):,}  class counts {np.bincount(y_test).tolist()}")
print("Train period:", df[TIME_COL].iloc[0], "->", df[TIME_COL].iloc[n_train - 1])
print("Test period: ", df[TIME_COL].iloc[n_train], "->", df[TIME_COL].iloc[-1])

## 4. Time-series random forest with grid search

The grid has 2 × 3 × 3 × 2 = 36 combinations × 4 time-series splits = 144 fits, which can take several minutes.

In [ ]:
rf = RandomForestClassifier(
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED,
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=TimeSeriesSplit(n_splits=4),
    n_jobs=1,          # the forest itself already uses all cores
    verbose=1,
)

print("Running GridSearchCV for the time-series random forest...")
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1-score: {grid.best_score_:.4f}")

best_model = grid.best_estimator_

## 5. Evaluation on the chronological test set

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

results_df = pd.DataFrame({
    "ACC": [accuracy_score(y_test, y_pred)],
    "F1": [f1_score(y_test, y_pred, zero_division=0)],
    "AUC": [roc_auc_score(y_test, y_proba)],
    "Sensitivity": [recall_score(y_test, y_pred, zero_division=0)],
    "Precision": [precision_score(y_test, y_pred, zero_division=0)],
    "Specificity": [tn / (tn + fp) if (tn + fp) else np.nan],
    "Kappa": [cohen_kappa_score(y_test, y_pred)],
    "TN": [tn], "FP": [fp], "FN": [fn], "TP": [tp],
}, index=["TimeSeries_RF_full"])
display(results_df.round(4))

print(classification_report(y_test, y_pred, digits=4, zero_division=0))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion matrix: time-series RF (full data)")
plt.tight_layout()
plt.show()

## 6. Which signals matter? Feature importances

If one or two signals dominate completely, check whether they are almost a copy of the label (see the limitations).

In [ ]:
importances = pd.Series(
    best_model.feature_importances_, index=feature_names
).sort_values(ascending=False)

display(importances.head(15).round(4).to_frame("importance"))

top15 = importances.head(15)
plt.figure(figsize=(7, 5))
plt.barh(top15.index[::-1], top15.values[::-1])
plt.xlabel("Importance")
plt.title("Top 15 feature importances: time-series RF (full data)")
plt.tight_layout()
plt.show()

## 7. Save the results locally

Written to `../outputs/` (not for upload).

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_out = OUTPUT_DIR / "timeseries_rf_full_metrics.csv"
importance_out = OUTPUT_DIR / "timeseries_rf_full_feature_importance.csv"

results_df.to_csv(metrics_out)
importances.to_frame("importance").to_csv(importance_out)

for path in (metrics_out, importance_out):
    print("Saved:", path)

## 8. Summary and conclusion

This variant checks whether the random forest also works on the full, unscaled signal set without the piece counter. As in notebook 06 (GridSearch), the validation design respects the time order.

No reference results are listed here: the numbers from the earlier version are invalid, because the target was among the features. Run the notebook on the downloaded dataset to obtain the real values, compare the cross-validated F1-score (Section 4) with the test F1-score (Section 5), and check the feature importances (Section 6) before interpreting a very high score.